# PPT 6-12 ??: KB??? ??? RAG ??

? ???? PPT 6-11? ??? ???? ??? ?? ??? ?????.

- PDF ?? ? ?? ??
- Chroma ????? ?? ?? ???
- LangChain RAG ?? ??
- PPT 12? ??? ???? ?? ?? ??


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
load_dotenv(r"C:\RAG\data\.env")

BASE_DIR = Path.cwd()
DATA_DIR = Path(r"C:\RAG\data")
PDF_CANDIDATES = sorted(DATA_DIR.glob("*.pdf"))
CHROMA_DIR = BASE_DIR / "chroma_db"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not PDF_CANDIDATES:
    raise FileNotFoundError(f"PDF ??? ?? ? ????: {DATA_DIR}")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY? ???? ?????. C:/RAG/data/.env? ?????.")

PDF_PATH = PDF_CANDIDATES[0]
print(f"PDF_PATH = {PDF_PATH}")
print(f"CHROMA_DIR = {CHROMA_DIR}")


PDF_PATH = C:\RAG\data\2024_KB_부동산_보고서_최종.pdf
CHROMA_DIR = C:\RAG\ch06\chroma_db


## PPT 6: PDF ?? ? ?? ??

In [3]:
def process_pdf(pdf_path: Path):
    loader = PyPDFLoader(str(pdf_path))
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)
    return documents, chunks


documents, chunks = process_pdf(PDF_PATH)
print(f"?? ??? ?: {len(documents)}")
print(f"?? ?: {len(chunks)}")
print(chunks[0].page_content[:300])

?? ??? ?: 84
?? ?: 135
1 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
 
 
Executive Summary 1 
 2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목 
주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정 
가능성이 큰 상황이다. 매수 수요 위축으로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 
시각이 팽배하다. 무엇보다 여전히 높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리 
부담은 주택 수요를 크게 위축시


## PPT 7: Chroma ????? ?? ?? ???

In [12]:
def initialize_vectorstore(chunks, persist_directory: Path):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)

    if persist_directory.exists() and any(persist_directory.iterdir()):
        vectorstore = Chroma(
            persist_directory=str(persist_directory),
            embedding_function=embeddings,
        )
        print("?? Chroma DB? ??????.")
    else:
        persist_directory.mkdir(parents=True, exist_ok=True)
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=str(persist_directory),
        )
        print("? Chroma DB? ??????.")

    return vectorstore


vectorstore = initialize_vectorstore(chunks, CHROMA_DIR)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

?? Chroma DB? ??????.


## PPT 8-11: RAG ??? ?? ?? ?? ??

In [5]:
store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def initialize_chain():
    template = """??? KB??? ??? ??????.
??? ????? ???? ??? ?????.
??? ???? ???, ??? ?? ?? ??? ??? ??????.

????:
{context}
"""

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", template),
            ("placeholder", "{chat_history}"),
            ("human", "{question}"),
        ]
    )

    model = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )

    base_chain = (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["question"]))
        )
        | prompt
        | model
        | StrOutputParser()
    )

    return RunnableWithMessageHistory(
        base_chain,
        get_session_history,
        input_messages_key="question",
        history_messages_key="chat_history",
    )


chain = initialize_chain()

NameError: name 'OPENAI_API_KEY' is not defined

In [6]:
def ask(question: str, session_id: str = "ppt12_demo"):
    response = chain.invoke(
        {"question": question},
        config={"configurable": {"session_id": session_id}},
    )
    print(f"???: {question}")
    print(f"??: {response}")
    print("-" * 80)
    return response

## PPT 12? ??? ?? ?? ??
?? ?? ???? ?? ??? ??? ? ?? ????? ?????.

In [7]:
questions = [
    "KB??? ???? ?? ??? ????.",
    "??? ??? ?? ??? ?? ????.",
    "?? ??? ???? ????? ??? ?? ????.",
]

for question in questions:
    ask(question)

???: KB??? ???? ?? ??? ????.
??: ????? ??? ??? ??? ????. ??? ??? ??? ??? ??? ??? ????. ??? ??? ??? ??? ??? ??? ????.
--------------------------------------------------------------------------------


???: ??? ??? ?? ??? ?? ????.
??: ??? ??? ?? ??? ?? ????. ??? ??? ??? ??? ??? ??? ???. ??? ??? ??? ??? ??? ??? ???.
--------------------------------------------------------------------------------


???: ?? ??? ???? ????? ??? ?? ????.
??: ????? ??? ??? ??? ??? ???. ??? ??? ??? ??? ??? ??? ???. ??? ??? ??? ??? ??? ??? ???.
--------------------------------------------------------------------------------


## ?? ??: Streamlit ??
??? ??? ??? ????? ?? ??? ?????.

```powershell
C:\RAG\venv\Scripts\python.exe -m streamlit run C:\RAG\ch06\app.py
```